In [13]:
import pandas as pd
import numpy as np

original_data_path = "../data/raw/"
data_binary_representation ="../data/processed/"

In [14]:
df = pd.read_csv(original_data_path + "wells.csv")
df_concentration = pd.read_csv(original_data_path + "concentration.csv")
df_head = pd.read_csv(original_data_path + "head.csv")
df_cores = pd.read_csv(original_data_path + "cores.csv")
df_pumping = pd.read_csv(original_data_path + "pumping.csv")

Сначала удалим все nan значение, затем удалим каскадно из других таблиц, после удалим дубликаты с округлением по x и y до целых и восстановим z.

In [72]:
df.loc[df['x'] == -999.0, "x"] = np.nan

nan_values_mask = df[['x','y','z']].isna().any(axis=1)

df_deleted = df[nan_values_mask]
df_deleted = df_deleted["well_id"].unique()
df = df[~nan_values_mask]

df["round_x"] = df["x"].round(0)
df["round_y"] = df["y"].round(0)

duplicated_mask = df.duplicated(subset=["round_x", "round_y", "z"], keep="first")
duplicated_idx = df[duplicated_mask]["well_id"].unique()

df = df[~duplicated_mask]
df = df.drop(columns=["round_x", "round_y"])

In [64]:
df_concentration = df_concentration[~df_concentration["well_id"].isin(df_deleted)]
df_head = df_head[~df_head["well_id"].isin(df_deleted)]
df_cores = df_cores[~df_cores["well_id"].isin(df_deleted)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(df_deleted)]

df_concentration = df_concentration[~df_concentration["well_id"].isin(duplicated_idx)]
df_head = df_head[~df_head["well_id"].isin(duplicated_idx)]
df_cores = df_cores[~df_cores["well_id"].isin(duplicated_idx)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(duplicated_idx)]

In [65]:
df.loc[(df["z"] > 5) & (df["z"] < 150), "z"] = -df["z"]

In [66]:
df.to_parquet(f"{data_binary_representation}wells.parquet", engine='pyarrow')

In [58]:
df_cores.loc[df_cores["z_top"] > 0, "z_top"] = -df_cores["z_top"]

df_cores = df_cores.sort_values(by=["well_id", "z_top"], ascending=[True, False])
df_cores["z_bottom"] = df_cores["z_bottom"].fillna(df_cores.groupby("well_id")["z_top"].shift(-1))

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom", "lithology"], na_position="last")
df_cores = df_cores.drop_duplicates(subset=["well_id", "z_top", "z_bottom"], keep="first")

df_cores.loc[df_cores["lithology"].isin(["Clay", "Caly"]), "lithology"] = "clay"
df_cores.loc[df_cores["lithology"].isin(["Gravel"]), "lithology"] = "gravel"
df_cores.loc[df_cores["lithology"].isin(["Loam"]), "lithology"] = "loam"
df_cores.loc[df_cores["lithology"].isin(["Sand", "SAND", "sand "]), "lithology"] = "sand"

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom"], ascending=[True, False, True])
prev_max_deph = df_cores.groupby("well_id")["z_bottom"].cummin().shift(1)
same_well = df_cores["well_id"] == df_cores["well_id"].shift(1)
is_inner = same_well & (df_cores["z_bottom"] >= prev_max_deph)
df_cores = df_cores[~is_inner]

df["round_x"] = df["x"].round(0)
df["round_y"] = df["y"].round(0)
duplicated_wells = df.duplicated(subset=["round_x", "round_y"], keep=False)
wells_names = df[duplicated_wells].copy()

depths = df_cores.groupby("well_id")["z_bottom"].min().reset_index()
merged_df = wells_names.merge(depths, on="well_id")
trust_index = merged_df.groupby(by=["round_x", "round_y"])["z_bottom"].idxmin()
trusted_wells = merged_df.loc[trust_index, "well_id"]

bad_wells = set(wells_names["well_id"]) - set(trusted_wells)
df_cores = df_cores[~df_cores["well_id"].isin(bad_wells)]

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom"], ascending=[True, False, True])
prev_z_bottom = df_cores.groupby("well_id")["z_bottom"].shift(1)
df_cores["z_top"] = np.minimum(df_cores["z_top"], prev_z_bottom.fillna(df_cores["z_top"]))

df_cores = df_cores.drop_duplicates(subset=["well_id", "z_top", "z_bottom", "lithology"], keep="first")

df_cores = df_cores.dropna(subset=["z_bottom", "lithology"])

In [60]:
df_cores.to_parquet(f"{data_binary_representation}cores.parquet")

In [75]:
df_head = df_head[df_head["head"] >= 0]

df_head = df_head.dropna(subset=["t", "head"])
df_head = df_head.drop_duplicates(subset=["well_id", "t"], keep="first")

df_head["t_datetime"] = pd.to_datetime(df_head["t"], format="%d.%m.%Y", errors="coerce")
bad_values = (
    (df_head["t_datetime"].isna()) |
    (df_head["t_datetime"] < "2020-01-01") |
    (df_head["t_datetime"] > "2025-10-31")
)
df_head = df_head[~bad_values]

In [76]:
df_head.to_parquet(f"{data_binary_representation}head.parquet")

In [15]:
df_pumping.loc[df_pumping["units"] == "l/s", "pumping_rate"] = (df_pumping["pumping_rate"] * 24 * 60 * 60 / 1000)
df_pumping.loc[df_pumping["units"] == "l/s", "units"] = "m3/day"

df_pumping = df_pumping.dropna(subset=["pumping_rate"])

df_pumping["t_datetime"] = pd.to_datetime(df_pumping["t"], format="%d.%m.%Y", errors="coerce")
bad_values = (
    (df_pumping["t_datetime"].isna()) |
    (df_pumping["t_datetime"] < "2020-01-01") |
    (df_pumping["t_datetime"] > "2025-12-01")
)
df_pumping = df_pumping[~bad_values]

df_pumping = df_pumping.drop(columns=["t_datetime"])

df_pumping.loc[df_pumping["pumping_rate"] < 0, "pumping_rate"] = -df_pumping["pumping_rate"]

In [16]:
df_pumping.to_parquet(f"{data_binary_representation}pumping.parquet", engine="pyarrow")